# Notebook 05 — Training (Clean Version)
## Adaptive Reliability-Aware Fusion (ARAF) Project

All fixes incorporated:
- Device handling via next(self.parameters()).device
- Two-level learning rates (encoder vs head)
- Pre-tokenization for speed
- num_workers=0 (Windows compatible)
- Overlap verification before training
- Nuclear reset pattern

---


## 1. Imports and setup

In [ ]:
import os, sys, json, copy, random, time
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import BertTokenizer
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224
MAX_TEXT_LEN  = 32

clean_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Imports successful.")


## 2. Reload all components from .py files

In [ ]:
@dataclass
class MultimodalSample:
    image: torch.Tensor
    text_ids: torch.Tensor
    attention_mask: torch.Tensor
    label: torch.Tensor
    raw_image: Optional[object] = None
    raw_text: str = ""
    dataset_name: str = "vqa_v2"
    sample_id: str = ""
    image_corrupted: bool = False
    text_corrupted: bool = False
    image_missing: bool = False
    text_missing: bool = False
    corruption_severity: float = 0.0

import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

baselines   = load_module("baselines",  "models/baselines.py")
araf_module = load_module("araf",       "models/araf.py")
corr_module = load_module("corruption", "corruption/corruption_module.py")

UnimodalImageModel = baselines.UnimodalImageModel
UnimodalTextModel  = baselines.UnimodalTextModel
NaiveFusionModel   = baselines.NaiveFusionModel
vqa_loss           = baselines.vqa_loss
vqa_accuracy       = baselines.vqa_accuracy
ARAFModel          = araf_module.ARAFModel
CorruptionModule   = corr_module.CorruptionModule

print("All modules loaded.")


## 3. Load tokenizer and data

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
with open("answer_vocab.json") as f:
    answer2idx = json.load(f)
idx2answer  = {v: k for k, v in answer2idx.items()}
NUM_CLASSES = len(answer2idx)

print("Loading VQA v2...")
hf_val_full = load_dataset("lmms-lab/VQAv2", split="validation",
                           trust_remote_code=True)
split    = hf_val_full.train_test_split(test_size=0.2, seed=42)
hf_train = split["train"]
hf_val   = split["test"]
print(f"Train: {len(hf_train):,}  Val: {len(hf_val):,}")


## 4. Optimized VQACorruptionDataset

Pre-tokenizes all text at load time (once) instead of per sample (every epoch).
This is the biggest single speedup — tokenization was the main CPU bottleneck.
Images still load on-the-fly (too large to pre-cache).
Corruption still applies on-the-fly (must be dynamic per epoch).


In [ ]:
class VQACorruptionDataset(Dataset):
    def __init__(self, hf_split, tokenizer, answer2idx,
                 corruption_module=None, max_samples=None, num_classes=3129):
        self.data = hf_split
        self.a2i  = answer2idx
        self.corr = corruption_module
        self.n_cls= num_classes

        if max_samples:
            self.data = self.data.select(range(min(max_samples, len(self.data))))

        # Pre-tokenize all questions ONCE at load time
        print(f"Pre-tokenizing {len(self.data):,} samples...")
        questions = [self.data[i]["question"] for i in range(len(self.data))]
        encoded   = tokenizer(questions, padding="max_length",
                              max_length=MAX_TEXT_LEN, truncation=True,
                              return_tensors="pt")
        self.all_text_ids   = encoded["input_ids"]
        self.all_attn_masks = encoded["attention_mask"]

        # Pre-compute all soft labels ONCE
        print("Pre-computing labels...")
        labels = []
        for i in range(len(self.data)):
            label = torch.zeros(num_classes)
            cnt = Counter(a["answer"].lower().strip()
                          for a in self.data[i]["answers"])
            for ans, c in cnt.items():
                if ans in answer2idx:
                    label[answer2idx[ans]] = min(c / 3.0, 1.0)
            labels.append(label)
        self.all_labels = torch.stack(labels)

        print(f"Dataset ready: {len(self.data):,} samples, "
              f"corruption={'ON' if corruption_module else 'OFF'}")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        # Image: still loaded on-the-fly
        pil = self.data[idx]["image"].convert("RGB")
        img = clean_transform(pil)

        # Text + label: instant lookup from pre-computed tensors
        sample = MultimodalSample(
            image=img,
            text_ids=self.all_text_ids[idx],
            attention_mask=self.all_attn_masks[idx],
            label=self.all_labels[idx],
            raw_text=self.data[idx]["question"],
            sample_id=str(self.data[idx].get("question_id", idx)),
        )
        if self.corr:
            sample = self.corr(sample)

        return {
            "image"             : sample.image,
            "text_ids"          : sample.text_ids,
            "attention_mask"    : sample.attention_mask,
            "label"             : sample.label,
            "image_corrupted"   : sample.image_corrupted,
            "text_corrupted"    : sample.text_corrupted,
            "image_missing"     : sample.image_missing,
            "text_missing"      : sample.text_missing,
            "corruption_severity": sample.corruption_severity,
            "raw_text"          : sample.raw_text,
            "sample_id"         : sample.sample_id,
        }

print("VQACorruptionDataset defined.")


## 5. Configuration and dataset building

Adjust MAX_TRAIN_SAMPLES and NUM_EPOCHS here.
For a quick test: 5000 samples, 5 epochs (~15 minutes).
For overnight: 50000 samples, 15 epochs (~6-8 hours with unfrozen encoders).


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
MAX_TRAIN_SAMPLES     = 10000   # increase for better results
MAX_VAL_SAMPLES       = 2000
BATCH_SIZE            = 32
NUM_EPOCHS            = 10
LEARNING_RATE_HEAD    = 3e-4    # fast for classifier/fusion/reliability
LEARNING_RATE_ENCODER = 1e-5    # slow for ResNet and BERT
LAMBDA_REG            = 0.1
FREEZE_ENCODERS       = False   # True = frozen (fast, lower accuracy)
                                # False = unfrozen (slow, higher accuracy)

print("Configuration:")
print(f"  MAX_TRAIN_SAMPLES : {MAX_TRAIN_SAMPLES}")
print(f"  MAX_VAL_SAMPLES   : {MAX_VAL_SAMPLES}")
print(f"  BATCH_SIZE        : {BATCH_SIZE}")
print(f"  NUM_EPOCHS        : {NUM_EPOCHS}")
print(f"  LR_HEAD           : {LEARNING_RATE_HEAD}")
print(f"  LR_ENCODER        : {LEARNING_RATE_ENCODER}")
print(f"  FREEZE_ENCODERS   : {FREEZE_ENCODERS}")
print(f"  DEVICE            : {DEVICE}")

# ── Corruption module ─────────────────────────────────────────────────────────
train_corruption = CorruptionModule(
    p_corrupt_image=0.5, p_corrupt_text=0.5,
    p_missing_image=0.1, p_missing_text=0.1,
    severity=None,
)

# ── Build datasets ────────────────────────────────────────────────────────────
train_dataset = VQACorruptionDataset(
    hf_train, tokenizer, answer2idx,
    corruption_module=train_corruption,
    max_samples=MAX_TRAIN_SAMPLES,
    num_classes=NUM_CLASSES,
)
val_dataset = VQACorruptionDataset(
    hf_val, tokenizer, answer2idx,
    corruption_module=None,
    max_samples=MAX_VAL_SAMPLES,
    num_classes=NUM_CLASSES,
)

def collate_fn(batch):
    return {
        "image"             : torch.stack([b["image"] for b in batch]),
        "text_ids"          : torch.stack([b["text_ids"] for b in batch]),
        "attention_mask"    : torch.stack([b["attention_mask"] for b in batch]),
        "label"             : torch.stack([b["label"] for b in batch]),
        "image_corrupted"   : [b["image_corrupted"] for b in batch],
        "text_corrupted"    : [b["text_corrupted"] for b in batch],
        "image_missing"     : [b["image_missing"] for b in batch],
        "text_missing"      : [b["text_missing"] for b in batch],
        "corruption_severity": [b["corruption_severity"] for b in batch],
        "raw_text"          : [b["raw_text"] for b in batch],
        "sample_id"         : [b["sample_id"] for b in batch],
    }

# num_workers=0 required on Windows (avoids multiprocessing deadlock)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False, collate_fn=collate_fn)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


## 6. Build models

In [ ]:
def build_models():
    """Build all four models. FREEZE_ENCODERS controls frozen/unfrozen."""
    return {
        "UnimodalImage": UnimodalImageModel(
            num_classes=NUM_CLASSES,
            frozen_encoder=FREEZE_ENCODERS).to(DEVICE),
        "UnimodalText": UnimodalTextModel(
            num_classes=NUM_CLASSES,
            frozen_encoder=FREEZE_ENCODERS).to(DEVICE),
        "NaiveFusion": NaiveFusionModel(
            num_classes=NUM_CLASSES,
            frozen_encoders=FREEZE_ENCODERS).to(DEVICE),
        "ARAF": ARAFModel(
            num_classes=NUM_CLASSES, fusion_dim=512,
            frozen_encoders=FREEZE_ENCODERS,
            lambda_reg=LAMBDA_REG).to(DEVICE),
    }

models = build_models()
print("Models built:")
for name, model in models.items():
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {name:<20}: {trainable:>12,} trainable / {total:>12,} total")


## 7. Build optimizers with two-level learning rates

In [ ]:
def build_optimizers(models):
    """
    Two learning rate groups:
    - Encoder weights (ResNet/BERT): slow lr to preserve pretrained knowledge
    - Head weights (classifier/fusion/reliability): fast lr for random-init layers
    """
    optimizers = {}
    schedulers = {}
    for name, model in models.items():
        encoder_params = [p for n, p in model.named_parameters()
                         if p.requires_grad and
                         any(k in n for k in ["encoder", "bert"])]
        head_params    = [p for n, p in model.named_parameters()
                         if p.requires_grad and
                         not any(k in n for k in ["encoder", "bert"])]
        opt = torch.optim.Adam([
            {"params": encoder_params, "lr": LEARNING_RATE_ENCODER,
             "weight_decay": 1e-4},
            {"params": head_params,    "lr": LEARNING_RATE_HEAD,
             "weight_decay": 1e-4},
        ])
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=NUM_EPOCHS, eta_min=1e-7)
        optimizers[name] = opt
        schedulers[name] = sch
        print(f"  {name:<20}: {len(encoder_params)} encoder params "
              f"(lr={LEARNING_RATE_ENCODER}), "
              f"{len(head_params)} head params (lr={LEARNING_RATE_HEAD})")
    return optimizers, schedulers

optimizers, schedulers = build_optimizers(models)


## 8. Verify optimizer overlap (CRITICAL)

This check must pass before training. If any overlap is 0,
the optimizer is not linked to the model and training will produce
flat loss — exactly the bug we hit multiple times today.


In [ ]:
print("Verifying optimizer-model parameter linkage...")
all_ok = True
for name, model in models.items():
    model_params = set(id(p) for p in model.parameters() if p.requires_grad)
    opt_params   = set(id(p) for group in optimizers[name].param_groups
                       for p in group["params"])
    overlap = len(model_params & opt_params)
    status  = "OK" if overlap > 0 else "FAIL"
    print(f"  {name:<20}: overlap={overlap} [{status}]")
    if overlap == 0:
        all_ok = False

if all_ok:
    print("All overlaps correct. Safe to train.")
else:
    print("ERROR: Some overlaps are 0. Rebuild models and optimizers before training.")


## 9. Training and validation step functions

In [ ]:
def train_step(model, batch, optimizer, model_name):
    model.train()
    # Move all tensors to device
    batch["image"]          = batch["image"].to(DEVICE)
    batch["text_ids"]       = batch["text_ids"].to(DEVICE)
    batch["attention_mask"] = batch["attention_mask"].to(DEVICE)
    batch["label"]          = batch["label"].to(DEVICE)

    if model_name == "ARAF":
        output = model(batch)
        losses = model.compute_loss(batch, output)
        loss   = losses["total_loss"]
        task_l = losses["task_loss"].item()
        reg_l  = losses["reg_loss"].item()
    else:
        output = model(batch)
        loss   = vqa_loss(output["logits"], batch["label"])
        task_l = loss.item()
        reg_l  = 0.0

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], max_norm=1.0)
    optimizer.step()

    with torch.no_grad():
        acc = vqa_accuracy(output["logits"].detach(), batch["label"])

    return {"total_loss": loss.item(), "task_loss": task_l,
            "reg_loss": reg_l, "accuracy": acc}


def val_step(model, batch, model_name):
    model.eval()
    batch["image"]          = batch["image"].to(DEVICE)
    batch["text_ids"]       = batch["text_ids"].to(DEVICE)
    batch["attention_mask"] = batch["attention_mask"].to(DEVICE)
    batch["label"]          = batch["label"].to(DEVICE)

    with torch.no_grad():
        if model_name == "ARAF":
            output = model(batch)
            losses = model.compute_loss(batch, output)
            loss   = losses["total_loss"].item()
        else:
            output = model(batch)
            loss   = vqa_loss(output["logits"], batch["label"]).item()
        acc = vqa_accuracy(output["logits"], batch["label"])

    return {"loss": loss, "accuracy": acc}

print("Step functions defined.")


## 10. Reset history (run before every training run)

In [ ]:
history = {name: {
    "train_loss": [], "train_acc": [],
    "val_loss":   [], "val_acc":  [],
    "reg_loss":   [],
} for name in models}
best_val_acc   = {name: 0.0 for name in models}
best_ckpt_path = {name: f"checkpoints/{name}_best.pt" for name in models}
os.makedirs("checkpoints", exist_ok=True)
print("History reset. Ready to train.")


## 11. Training loop

Run this cell and leave it. Do not interrupt mid-epoch.
Watch batch 20 output — loss should start ~0.4 and drop fast.
If loss starts at ~0.006 and stays flat, stop and check overlap.


In [ ]:
print("Starting training...")
print("=" * 70)

total_start = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    train_stats = {name: defaultdict(list) for name in models}
    val_stats   = {name: defaultdict(list) for name in models}

    # Training
    for batch_idx, batch in enumerate(train_loader):
        for name, model in models.items():
            step_out = train_step(model, batch, optimizers[name], name)
            for k, v in step_out.items():
                train_stats[name][k].append(v)

        if (batch_idx + 1) % 20 == 0:
            araf_loss = np.mean(train_stats["ARAF"]["total_loss"])
            araf_acc  = np.mean(train_stats["ARAF"]["accuracy"])
            print("  Epoch {}/{} Batch {}/{} | ARAF loss={:.4f} acc={:.4f}".format(
                epoch+1, NUM_EPOCHS, batch_idx+1, len(train_loader),
                araf_loss, araf_acc))

    # Validation
    for batch in val_loader:
        for name, model in models.items():
            step_out = val_step(model, batch, name)
            for k, v in step_out.items():
                val_stats[name][k].append(v)

    # Schedulers
    for name in models:
        schedulers[name].step()

    # Log
    epoch_time = time.time() - epoch_start
    print("
Epoch {}/{} completed in {:.1f}s".format(
        epoch+1, NUM_EPOCHS, epoch_time))
    print("{:<20} {:>10} {:>10} {:>10} {:>10}".format(
        "Model", "Train Loss", "Train Acc", "Val Loss", "Val Acc"))
    print("-" * 60)

    for name in models:
        tr_loss = np.mean(train_stats[name]["total_loss"])
        tr_acc  = np.mean(train_stats[name]["accuracy"])
        vl_loss = np.mean(val_stats[name]["loss"])
        vl_acc  = np.mean(val_stats[name]["accuracy"])
        reg_l   = np.mean(train_stats[name]["reg_loss"]) if name=="ARAF" else 0.0

        history[name]["train_loss"].append(tr_loss)
        history[name]["train_acc"].append(tr_acc)
        history[name]["val_loss"].append(vl_loss)
        history[name]["val_acc"].append(vl_acc)
        history[name]["reg_loss"].append(reg_l)

        print(f"{name:<20} {tr_loss:>10.4f} {tr_acc:>10.4f} "
              f"{vl_loss:>10.4f} {vl_acc:>10.4f}")

        if vl_acc > best_val_acc[name]:
            best_val_acc[name] = vl_acc
            torch.save({
                "epoch": epoch+1, "model_name": name,
                "state_dict": model.state_dict(),
                "val_acc": vl_acc, "val_loss": vl_loss,
            }, best_ckpt_path[name])
            print(f"  -> New best for {name}: val_acc={vl_acc:.4f} (saved)")
    print()

total_time = time.time() - total_start
print(f"Training complete in {total_time/60:.1f} minutes.")
print("Best validation accuracies:")
for name, acc in best_val_acc.items():
    print(f"  {name:<20}: {acc:.4f}")


## 12. Save models and training history

In [ ]:
# Save model weights for Notebook 06
for name, model in models.items():
    path = f"checkpoints/{name}_eval.pt"
    torch.save(model.state_dict(), path)
    print(f"Saved: {path}")

# Save training history
history_serializable = {
    name: {k: [float(v) for v in vals] for k, vals in h.items()}
    for name, h in history.items()
}
with open("training_history.json", "w") as f:
    json.dump(history_serializable, f, indent=2)
print("Saved: training_history.json")


## 13. Training curves

In [ ]:
colors = {"UnimodalImage":"#7F77DD","UnimodalText":"#1D9E75",
          "NaiveFusion":"#D85A30","ARAF":"#185FA5"}
epochs = list(range(1, NUM_EPOCHS+1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name in models:
    axes[0].plot(epochs, history[name]["train_loss"],
                 color=colors[name], label=name, linewidth=2)
axes[0].set_title("Training loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

for name in models:
    axes[1].plot(epochs, history[name]["val_acc"],
                 color=colors[name], label=name, linewidth=2, marker="o")
axes[1].set_title("Validation accuracy (clean)")
axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

task_losses = [t - r for t, r in zip(history["ARAF"]["train_loss"],
                                      history["ARAF"]["reg_loss"])]
axes[2].plot(epochs, history["ARAF"]["train_loss"],
             color="#185FA5", label="Total loss", linewidth=2)
axes[2].plot(epochs, task_losses,
             color="#185FA5", label="Task loss", linewidth=2, linestyle="--")
axes[2].plot(epochs, history["ARAF"]["reg_loss"],
             color="#FAC775", label="Reg loss", linewidth=2, linestyle=":")
axes[2].set_title("ARAF loss decomposition")
axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle("Training curves", fontsize=14)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")


## 14. Nuclear reset (use if something goes wrong)

Run this cell if you need to start completely fresh mid-session.
Always run the overlap check after this before training.


In [ ]:
# Nuclear reset — fresh models, fresh optimizers, clean history
models     = build_models()
optimizers, schedulers = build_optimizers(models)

# Verify overlap
print("Overlap check after nuclear reset:")
for name, model in models.items():
    model_params = set(id(p) for p in model.parameters() if p.requires_grad)
    opt_params   = set(id(p) for group in optimizers[name].param_groups
                       for p in group["params"])
    overlap = len(model_params & opt_params)
    print(f"  {name}: overlap={overlap}")

# Reset history
history = {name: {
    "train_loss": [], "train_acc": [],
    "val_loss":   [], "val_acc":  [],
    "reg_loss":   [],
} for name in models}
best_val_acc = {name: 0.0 for name in models}
print("Nuclear reset complete. Safe to train.")
